In [6]:
using Random, StatsBase, Distributions

In [15]:
# Function generate_instance

function generate_instance(n_zones, n_centers_locations, p, n_types, n_services, n_periods, n_hours, seed)

    Random.seed!(seed)

    # If n_hours == 0 ; generate random working time 6h, 7h, 8h or 9h
    if n_hours == 0
        hours = sample([6, 7, 8, 9], Weights([0.20, 0.30, 0.30, 0.20]), n_types)
    else
        hours = fill(n_hours, n_types)
    end

    # Lower bound for capacity to hire workers in each possible center location 
    # lower_w
    lower_w = ones(Int, n_centers_locations)
    # Upper bound for capacity to hire workers in each possible center location
    # upper_w
    upper_w = fill(n_types * n_periods * 5, n_centers_locations)

    # Skill matrix for workers : Lines : n_types ; Columns : n_services ; at least one "1" per column
    # skill_matrix
    skill_matrix = zeros(Int, n_types, n_services)

    for l in 1:n_services
        skill_matrix[rand(1:n_types), l] = 1
    end #at least one "1" per column

    for k in 1:n_types, l in 1:n_services
        if skill_matrix[k, l] == 0
            skill_matrix[k, l] = rand(0:1)
        end
    end


    # Cover matrix for centers : Lines : n_centers_locations ; Columns : n_zones ; at least one "1" per column
    # cover_matrix
    cover_matrix = zeros(Int, n_centers_locations, n_zones)
    for j in 1:n_zones 
        cover_matrix[rand(1:n_centers_locations),j] = 1
    end
    for i in 1:n_centers_locations, j in 1:n_zones
        if cover_matrix[i,j] == 0
            cover_matrix[i,j] = rand(0:1)
        end
    end


    # Demands : 3D matrix : n_zones*n_services*n_periods
    # demands
    demands = zeros(Int, n_zones, n_services, n_periods)

    lambdas = [3, 8, 15]     #rural area, middle, dense ; number of visit per period
    lambda_per_zone = sample(lambdas, n_zones)
    for j in 1:n_zones, l in 1:n_services, t in 1:n_periods
        demands[j, l, t] = rand(Poisson(lambda_per_zone[j]))
    end


    # Services times : matrix n_services*n_periods
    # servtime
    base = rand(LogNormal(-0.3, 0.3), n_services)          # initial service time for each service
    noise = rand(Normal(0, 0.05), n_services, n_periods)   # add a noise 
    servtime = round.(max.(base .+ noise, 0.33), digits=2) # to be positive
    
    # Cost to open each center : vector 
    # c_f
    c_f = rand(3000:10000, n_centers_locations)
    
    # Cost to hire worker type k in center i : matrix 
    # c_hire
    c_hire = rand(25:40, n_centers_locations, n_types)

    # Cost to assign worker type k to service l on period t at center i : 4D matrix
    # c_assig
    c_assig = rand(20:30, n_centers_locations, n_types, n_services, n_periods)

    # Cost per hour if worker type k provides service l on period t at center i : 4D matrix
    # c_hour
    c_hour = round.(rand(Float64, n_centers_locations, n_types, n_services, n_periods) * 10 .+ 10, digits=2)

    # Penality cost for idle hours for worker of type k on period t at center i : 3D matrix
    # c_over
    c_over  = rand(18:25, n_centers_locations, n_types, n_periods)


    # Penality cost for unmet demand of service l on period t in zone j : 3D matrix
    # c_unmet
    c_unmet = rand(40:50, n_zones, n_services, n_periods)

    return hours, lower_w, upper_w, skill_matrix, cover_matrix, demands, servtime, c_f, c_hire, c_assig, c_hour, c_over, c_unmet
end

generate_instance (generic function with 1 method)

In [10]:
using Dates

# Function save_instance
function save_instance(folder, n_zones, n_centers_locations, p, n_types, n_services, n_periods, seed, hours, 
                lower_w, upper_w, skill_matrix, cover_matrix, demands, servtime, c_f, c_hire, c_assig, c_hour, c_over, c_unmet)

    mkpath(folder) 

    filename = "instance_J$(n_zones)_p$(p)_I$(n_centers_locations)_K$(n_types)_L$(n_services)_T$(n_periods)_seed$(seed).dat"
    filepath = joinpath(folder, filename) 
       
    open(filepath, "w") do f
        # Metadata header
        println(f, "# n_zones              : $n_zones")
        println(f, "# n_centers_locations  : $n_centers_locations")
        println(f, "# p                    : $p")
        println(f, "# n_types              : $n_types")
        println(f, "# n_services           : $n_services")
        println(f, "# n_periods            : $n_periods")
        println(f, "# seed                 : $seed")
        println(f, "# generated            : $(Dates.today())")
        println(f, "# -------------------")


        # Data
        println(f, n_zones, " ", n_centers_locations, " ", p, " ", n_types, " ", n_services, " ", n_periods)

        println(f, "# Working hours")
        println(f, join(hours, " "))
        
        println(f,"# Capacity bounds for centers")
        for i in 1:n_centers_locations 
            println(f, lower_w[i], " ", upper_w[i])
        end

        
        # 2D matrix : one line per row
        println(f,"# Cover matrix")
        for i in 1:n_centers_locations
            println(f, join(cover_matrix[i, :], " "))
        end

        
        println(f,"# Skill matrix")
        for k in 1:n_types
            println(f, join(skill_matrix[k, :], " "))
        end

        println(f, "# Demands")
        for j in 1:n_zones
            for l in 1:n_services
                println(f, join(demands[j,l, :], " "))
            end
        end

        println(f, "# Service times")
        for l in 1:n_services 
            println(f, join(servtime[l, :], " "))
        end

        println(f, "# Opening costs")
        println(f, join(c_f, " "))

        println(f, "# Hiring costs")
        for i in 1:n_centers_locations 
            println(f, join(c_hire[i, :], " "))
        end

        println(f, "# Assignment costs")
        for i in 1:n_centers_locations
            for k in 1:n_types
                for l in 1:n_services
                    println(f, join(c_assig[i, k, l, :], " "))
                end
            end
        end
        
        println(f, "# Hourly costs")
        for i in 1:n_centers_locations
            for k in 1:n_types
                for l in 1:n_services
                    println(f, join(c_hour[i, k, l, :], " "))
                end
            end
        end
        
        println(f, "# Idle hours penalty costs")
        for i in 1:n_centers_locations
            for k in 1:n_types
                println(f, join(c_over[i, k, :], " "))
            end
        end
        
        println(f, "# Unmet demand penalty costs")
        for j in 1:n_zones
            for l in 1:n_services
                println(f, join(c_unmet[j, l, :], " "))
            end
        end
            
    end
        
end


save_instance (generic function with 1 method)

In [17]:
# Chosen data
n_zones = 3               #number of zones
n_centers_locations = 5  #number of possible centers locations
p = 1                    #minimum number of centers to open
n_types = 3              #number of different worker types
n_services = 4           #number of different proposed services
n_periods = 2            #number of periods in the planing horizon
n_hours = 0              #number of working hours per period ; if n_hours == 0, random

seed = 33

# Generate
hours, lower_w, upper_w, skill_matrix, cover_matrix, demand, servtime, c_f, 
c_hire, c_assig, c_hour, c_over, c_unmet = generate_instance(n_zones, n_centers_locations, p, n_types, 
                                                                n_services, n_periods, n_hours, seed)

# Save
save_instance("instances", n_zones, n_centers_locations, p, n_types, n_services, n_periods, seed, hours,
              lower_w, upper_w, skill_matrix, cover_matrix, demand, servtime, c_f, c_hire, c_assig, c_hour, 
                c_over, c_unmet)